# 01 - 线性回归：从一条直线理解机器学习

本 Notebook 根据 `03_machine_learning/01_linear_regression.py` 重新组织和扩充而来，目标不是“把脚本搬进 Notebook”，而是把线性回归讲成一个可以交互运行、逐步观察的教程。

你会学到：

1. 回归问题在做什么：从数据点中学习一个连续值预测函数
2. 线性模型为什么是 `ŷ = wx + b`，以及“线性”真正线性在哪里
3. 损失函数如何把“拟合得好不好”变成一个数字
4. 梯度下降如何一步步找到好参数
5. 正规方程如何一次性求出最小二乘解
6. 多元线性回归、特征缩放、评估指标、残差诊断
7. 过拟合、欠拟合、正则化和一个完整房价预测小项目

建议运行方式：从上到下逐个运行代码单元。遇到图表时先观察图，再读下面的解释。


In [ ]:
# 环境准备
import numpy as np
import matplotlib
matplotlib.use("Agg")  # 在脚本验证或无图形界面环境中也能运行
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline

plt.rcParams["font.sans-serif"] = [
    "WenQuanYi Micro Hei", "WenQuanYi Zen Hei", "Noto Sans CJK SC",
    "SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"
]
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)

print("环境准备完成：NumPy + Matplotlib + scikit-learn")


## 1. 回归问题：预测一个连续数值

机器学习中的监督学习可以粗略分为两类：

| 类型 | 目标 | 例子 |
|---|---|---|
| 回归 Regression | 预测连续数值 | 房价、销量、气温、点击率 |
| 分类 Classification | 预测离散类别 | 垃圾邮件/正常邮件、猫/狗、疾病阴性/阳性 |

线性回归是最基础的回归模型。它假设输入特征 `x` 和目标值 `y` 之间可以用一条直线近似：

$$\hat y = wx + b$$

其中：

- `x`：输入特征，例如房屋面积
- `ŷ`：模型预测值
- `w`：斜率/权重，表示 x 增加 1 单位时，预测值平均变化多少
- `b`：截距/偏置，表示 x=0 时的基础预测值

先用一个房价小例子建立直觉。


In [ ]:
# 一个简单的房价数据集：面积 -> 价格
area = np.array([45, 55, 60, 72, 85, 95, 110, 125, 140, 160], dtype=float)
price = np.array([142, 155, 178, 205, 238, 260, 318, 352, 390, 455], dtype=float)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(area, price, s=70, color="steelblue", edgecolor="white", label="历史成交数据")
ax.set_xlabel("面积 x（平方米）")
ax.set_ylabel("价格 y（万元）")
ax.set_title("回归问题：从面积预测房价")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

print("问题：如果一套房子面积为 100 平方米，它大约值多少钱？")


图中的点并不完全在一条直线上，因为真实世界有噪声：地段、楼层、装修、朝向都会影响房价。

线性回归做的事情是：找一条“总体上离这些点最近”的直线。

这句话有两个关键问题：

1. 什么叫“离得最近”？—— 需要损失函数。
2. 怎么找到这条直线？—— 可以用梯度下降或正规方程。


## 2. 模型假设：线性到底线性在哪里？

一元线性回归：

$$\hat y = wx + b$$

多元线性回归：

$$\hat y = w_1x_1 + w_2x_2 + \cdots + w_dx_d + b$$

注意：线性回归的“线性”主要指 **模型关于参数是线性的**。例如：

$$\hat y = w_1x + w_2x^2 + b$$

虽然它画出来可能是曲线，但对参数 `w1, w2, b` 仍然是线性的，所以仍然可以看作线性回归，只是用了多项式特征。


In [ ]:
# 同一批非线性数据，分别用 x 和 [x, x^2] 做线性回归
x_curve = np.linspace(-3, 3, 80)
y_curve = 1.5 * x_curve**2 - 2 * x_curve + 3 + rng.normal(0, 1.2, size=x_curve.size)

# 只用 x：直线模型
model_line = LinearRegression().fit(x_curve.reshape(-1, 1), y_curve)

# 用 x 和 x^2：参数仍线性，但特征经过了非线性变换
X_poly2 = np.column_stack([x_curve, x_curve**2])
model_poly2 = LinearRegression().fit(X_poly2, y_curve)

x_plot = np.linspace(-3.2, 3.2, 300)
y_line = model_line.predict(x_plot.reshape(-1, 1))
y_poly2 = model_poly2.predict(np.column_stack([x_plot, x_plot**2]))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_curve, y_curve, s=25, alpha=0.65, label="带噪声的数据")
ax.plot(x_plot, y_line, "r--", lw=2, label="只用 x：直线")
ax.plot(x_plot, y_poly2, "g", lw=2.5, label="用 x 和 x²：曲线")
ax.set_title("线性模型也可以通过特征变换拟合曲线")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

print("直线模型参数：", model_line.coef_, model_line.intercept_)
print("二次特征模型参数：[w_x, w_x2] =", model_poly2.coef_, "b =", round(model_poly2.intercept_, 4))


## 3. 损失函数：把“拟合得差不差”变成数字

如果预测值是 `ŷ`，真实值是 `y`，误差可以写成：

$$e_i = \hat y_i - y_i$$

常见损失函数：

| 名称 | 公式 | 直觉 | 特点 |
|---|---|---|---|
| MAE | $\frac{1}{n}\sum |\hat y_i-y_i|$ | 平均偏差多少 | 对异常值更稳健，但 0 点不可导 |
| MSE | $\frac{1}{n}\sum (\hat y_i-y_i)^2$ | 平均平方误差 | 可导、凸、最常用于线性回归 |
| RMSE | $\sqrt{MSE}$ | 与 y 同单位的误差 | 更容易解释 |

为什么不直接用误差和 $\sum(\hat y_i-y_i)$？因为正负误差会抵消。


In [ ]:
# 正负误差抵消的例子
true = np.array([10, 20, 30, 40])
pred = np.array([12, 18, 33, 37])
errors = pred - true

print("真实值：", true)
print("预测值：", pred)
print("误差：  ", errors)
print("误差和：", errors.sum(), "  <- 接近 0 不代表预测很好")
print("MAE：", np.mean(np.abs(errors)))
print("MSE：", np.mean(errors**2))
print("RMSE：", np.sqrt(np.mean(errors**2)))


In [ ]:
# 可视化 MAE 与 MSE 对大误差的不同态度
err = np.linspace(-10, 10, 400)
mae_loss = np.abs(err)
mse_loss = err**2

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(err, mae_loss, label="绝对误差 |e|", lw=2)
ax.plot(err, mse_loss, label="平方误差 e²", lw=2)
ax.set_xlabel("误差 e = ŷ - y")
ax.set_ylabel("单个样本的损失")
ax.set_title("MSE 会显著放大大误差")
ax.set_ylim(0, 45)
ax.grid(alpha=0.3)
ax.legend()
plt.show()


## 4. 参数空间：每一组 `(w, b)` 都对应一个损失

对一元线性回归来说，只要给定 `w` 和 `b`，模型就确定了：

$$\hat y_i = wx_i + b$$

于是 MSE 也确定了：

$$L(w,b)=\frac{1}{n}\sum_i(wx_i+b-y_i)^2$$

我们可以把 `L(w,b)` 看成一片地形：

- 横轴是 `w`
- 纵轴是 `b`
- 高度是损失 `L`

训练模型就是找到这片地形最低的地方。


In [ ]:
# 用房价数据观察不同 w,b 的损失地形
w_values = np.linspace(1.5, 3.5, 80)
b_values = np.linspace(0, 80, 80)
W, B = np.meshgrid(w_values, b_values)
Loss = np.zeros_like(W)

for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        y_hat = W[i, j] * area + B[i, j]
        Loss[i, j] = np.mean((y_hat - price) ** 2)

# sklearn 的最优参数作为参照
fit = LinearRegression().fit(area.reshape(-1, 1), price)
w_star, b_star = fit.coef_[0], fit.intercept_

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(1, 2, 1)
cont = ax1.contourf(W, B, Loss, levels=40, cmap="viridis")
ax1.plot(w_star, b_star, "r*", markersize=14, label="最小二乘解")
ax1.set_xlabel("w")
ax1.set_ylabel("b")
ax1.set_title("MSE 等高线：颜色越深损失越小")
ax1.legend()
fig.colorbar(cont, ax=ax1, label="MSE")

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.plot_surface(W, B, Loss, cmap="viridis", alpha=0.85, edgecolor="none")
ax2.scatter([w_star], [b_star], [np.min(Loss)], color="red", s=80)
ax2.set_xlabel("w")
ax2.set_ylabel("b")
ax2.set_zlabel("MSE")
ax2.set_title("损失地形：一个凸碗")
plt.tight_layout()
plt.show()

print(f"最小二乘解：w={w_star:.4f}, b={b_star:.4f}")


线性回归 + MSE 的损失地形是凸的，像一个碗。

这很重要：凸函数没有“假山谷”。只要优化方法足够合理，就可以到达全局最低点。


## 5. 梯度下降：沿着最陡下坡方向走

梯度表示函数上升最快的方向。要降低损失，就沿着负梯度方向更新参数：

$$w \leftarrow w - \alpha \frac{\partial L}{\partial w}$$

$$b \leftarrow b - \alpha \frac{\partial L}{\partial b}$$

其中 $\alpha$ 是学习率，控制每一步走多大。

对 MSE：

$$\frac{\partial L}{\partial w}=\frac{2}{n}\sum_i(\hat y_i-y_i)x_i$$

$$\frac{\partial L}{\partial b}=\frac{2}{n}\sum_i(\hat y_i-y_i)$$


In [ ]:
def mse_for_line(x, y, w, b):
    return np.mean((w * x + b - y) ** 2)

def gradients_for_line(x, y, w, b):
    n = len(x)
    pred = w * x + b
    error = pred - y
    dw = (2 / n) * np.sum(error * x)
    db = (2 / n) * np.sum(error)
    return dw, db

# 用数值差分检查解析梯度是否正确
w0, b0 = 1.0, 10.0
dw, db = gradients_for_line(area, price, w0, b0)
eps = 1e-5
num_dw = (mse_for_line(area, price, w0 + eps, b0) - mse_for_line(area, price, w0 - eps, b0)) / (2 * eps)
num_db = (mse_for_line(area, price, w0, b0 + eps) - mse_for_line(area, price, w0, b0 - eps)) / (2 * eps)

print(f"解析梯度 dw={dw:.6f}, 数值梯度 dw={num_dw:.6f}")
print(f"解析梯度 db={db:.6f}, 数值梯度 db={num_db:.6f}")
print("两者接近，说明梯度公式实现正确。")


In [ ]:
def train_line_gd(x, y, lr=0.05, epochs=400, w_init=0.0, b_init=0.0, standardize=True):
    """用梯度下降训练一元线性回归。

    为了让学习率更容易设置，默认先把 x 标准化：
        x_scaled = (x - mean) / std
    梯度下降在标准化空间中训练，最后再把参数换回原始 x 的尺度。
    """
    if standardize:
        x_mean, x_std = x.mean(), x.std()
        x_train = (x - x_mean) / x_std
    else:
        x_mean, x_std = 0.0, 1.0
        x_train = x

    w_scaled, b_scaled = w_init, b_init
    history = {"w": [], "b": [], "w_scaled": [], "b_scaled": [], "loss": []}

    for _ in range(epochs):
        dw, db = gradients_for_line(x_train, y, w_scaled, b_scaled)
        w_scaled -= lr * dw
        b_scaled -= lr * db

        # 换回原始 x 尺度，便于和 sklearn/正规方程比较
        w_original = w_scaled / x_std
        b_original = b_scaled - w_original * x_mean
        history["w"].append(w_original)
        history["b"].append(b_original)
        history["w_scaled"].append(w_scaled)
        history["b_scaled"].append(b_scaled)
        history["loss"].append(mse_for_line(x, y, w_original, b_original))

    return history["w"][-1], history["b"][-1], history

w_gd, b_gd, hist = train_line_gd(area, price, lr=0.05, epochs=400)

print(f"梯度下降结果：w={w_gd:.4f}, b={b_gd:.4f}, loss={hist['loss'][-1]:.2f}")
print(f"sklearn 最小二乘：w={w_star:.4f}, b={b_star:.4f}")

x_line = np.linspace(area.min() - 5, area.max() + 5, 200)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(area, price, s=60, label="数据")
axes[0].plot(x_line, w_gd * x_line + b_gd, "r", lw=2, label="梯度下降拟合")
axes[0].plot(x_line, w_star * x_line + b_star, "g--", lw=2, label="最小二乘解")
axes[0].set_title("拟合直线")
axes[0].set_xlabel("面积")
axes[0].set_ylabel("价格")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(hist["loss"], lw=1.8)
axes[1].set_yscale("log")
axes[1].set_title("训练损失下降（对数坐标）")
axes[1].set_xlabel("迭代次数")
axes[1].set_ylabel("MSE")
axes[1].grid(alpha=0.3)

axes[2].plot(hist["w"], label="w")
axes[2].plot(hist["b"], label="b")
axes[2].axhline(w_star, color="C0", ls="--", alpha=0.5)
axes[2].axhline(b_star, color="C1", ls="--", alpha=0.5)
axes[2].set_title("参数随训练变化")
axes[2].set_xlabel("迭代次数")
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()


### 5.1 学习率：走太小很慢，走太大可能发散

学习率不是模型学出来的参数，而是人为设置的超参数。

- 太小：每一步都很谨慎，训练慢
- 适中：稳定下降
- 太大：跳过谷底，甚至越跳越远

下面固定同一批数据，比较不同学习率的损失曲线。


In [ ]:
learning_rates = [0.001, 0.01, 0.05, 0.5, 1.2]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for lr in learning_rates:
    _, _, h = train_line_gd(area, price, lr=lr, epochs=120)
    loss = np.array(h["loss"])
    label = f"lr={lr:g}"
    axes[0].plot(loss, label=label)
    axes[1].plot(np.clip(loss, 1, 1e7), label=label)

axes[0].set_title("不同学习率的损失曲线")
axes[0].set_xlabel("迭代次数")
axes[0].set_ylabel("MSE")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].set_yscale("log")
axes[1].set_title("对数坐标：过大学习率会震荡或发散")
axes[1].set_xlabel("迭代次数")
axes[1].set_ylabel("MSE (log, clipped)")
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.show()


In [ ]:
# 把梯度下降路径叠加到损失等高线上
path_w = np.array(hist["w"])
path_b = np.array(hist["b"])

fig, ax = plt.subplots(figsize=(7, 5.5))
cont = ax.contourf(W, B, Loss, levels=40, cmap="viridis")
ax.plot(path_w[::80], path_b[::80], "w.-", lw=1.5, ms=5, label="梯度下降路径")
ax.plot(w_star, b_star, "r*", markersize=14, label="最小点")
ax.set_xlabel("w")
ax.set_ylabel("b")
ax.set_title("参数空间中的优化路径")
ax.legend()
fig.colorbar(cont, ax=ax, label="MSE")
plt.show()


## 6. 正规方程：不用迭代，一次求解

线性回归的最小二乘问题可以写成矩阵形式：

$$\hat y = X\theta$$

其中 `X` 是设计矩阵，第一列通常是全 1，对应偏置项。

当 $X^TX$ 可逆时，最优解是：

$$\theta = (X^TX)^{-1}X^Ty$$

实际代码中通常不直接求逆，而用 `np.linalg.lstsq` 或 `np.linalg.pinv`，数值上更稳定。


In [ ]:
# 设计矩阵：第一列为 1，第二列为面积
X_design = np.column_stack([np.ones_like(area), area])

theta_inv = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ price
theta_lstsq, residuals, rank, s = np.linalg.lstsq(X_design, price, rcond=None)

print("直接正规方程 theta=[b, w]：", theta_inv)
print("np.linalg.lstsq theta=[b, w]：", theta_lstsq)
print("sklearn intercept, coef：", fit.intercept_, fit.coef_[0])
print("矩阵秩 rank：", rank)


正规方程和梯度下降的对比：

| 方法 | 优点 | 缺点 | 常用场景 |
|---|---|---|---|
| 正规方程 / 最小二乘 | 不需要学习率；小数据很方便 | 特征非常多时矩阵计算昂贵；共线性会带来数值问题 | 小中型线性回归 |
| 梯度下降 | 可扩展到大数据和复杂模型 | 需要调学习率；需要迭代 | 大规模数据、深度学习 |


## 7. 模型评估：不要只看训练损失

训练完成后，常用指标包括：

- MSE：平方误差平均值，越小越好
- RMSE：MSE 开根号，单位与目标值相同
- MAE：绝对误差平均值，直观且对异常值稳健
- R²：模型解释了多少目标变量的变化，越接近 1 越好

$$R^2 = 1 - \frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}$$

如果 R² = 0，说明模型和“永远预测均值”差不多；如果 R² < 0，说明模型比预测均值还差。


In [ ]:
y_pred = fit.predict(area.reshape(-1, 1))
mse = mean_squared_error(price, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(price, y_pred)
r2 = r2_score(price, y_pred)

print(f"MSE  = {mse:.3f}")
print(f"RMSE = {rmse:.3f} 万元")
print(f"MAE  = {mae:.3f} 万元")
print(f"R²   = {r2:.4f}")

# 手动验证 R²
ss_res = np.sum((price - y_pred) ** 2)
ss_tot = np.sum((price - price.mean()) ** 2)
print(f"手动计算 R² = {1 - ss_res/ss_tot:.4f}")


In [ ]:
# 预测值-真实值图 + 残差图
residuals = price - y_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(price, y_pred, s=70, alpha=0.75)
lo = min(price.min(), y_pred.min())
hi = max(price.max(), y_pred.max())
axes[0].plot([lo, hi], [lo, hi], "r--", label="完美预测")
axes[0].set_xlabel("真实价格")
axes[0].set_ylabel("预测价格")
axes[0].set_title("预测值 vs 真实值")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].scatter(y_pred, residuals, s=70, alpha=0.75)
axes[1].axhline(0, color="r", ls="--")
axes[1].set_xlabel("预测价格")
axes[1].set_ylabel("残差 = 真实 - 预测")
axes[1].set_title("残差图：寻找未被模型捕捉的模式")
axes[1].grid(alpha=0.3)
plt.show()


残差图是线性回归诊断中非常重要的工具：

- 残差随机散在 0 附近：线性模型基本合理
- 残差呈 U 形：可能漏掉了非线性关系
- 残差随预测值变大而扩散：可能存在异方差
- 个别点残差特别大：可能是异常值或漏掉关键特征


## 8. 多元线性回归：多个特征一起预测

现实任务通常不只有一个特征。比如房价可能同时取决于：

- 面积
- 房间数
- 楼龄
- 距离地铁的距离
- 绿化率

多元线性回归写作：

$$\hat y = w_1x_1+w_2x_2+\cdots+w_dx_d+b$$

矩阵形式：

$$\hat y = Xw+b$$


In [ ]:
# 生成一个多特征房价数据集
n = 500
house = {
    "面积(m²)": rng.uniform(35, 180, n),
    "房间数": rng.integers(1, 6, n).astype(float),
    "楼龄(年)": rng.uniform(0, 40, n),
    "距地铁(km)": rng.uniform(0.2, 12, n),
    "绿化率(%)": rng.uniform(10, 60, n),
}
feature_names = list(house.keys())
X_house = np.column_stack([house[k] for k in feature_names])
true_coef = np.array([2.1, 8.0, -0.7, -2.8, 0.35])
true_intercept = 45.0
y_house = X_house @ true_coef + true_intercept + rng.normal(0, 12, n)

print("X_house shape:", X_house.shape)
print("前 3 行特征：")
for row in X_house[:3]:
    print(row)
print("房价范围：", round(y_house.min(), 1), "~", round(y_house.max(), 1), "万元")


In [ ]:
# EDA：每个特征与房价的关系
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, name in enumerate(feature_names):
    ax = axes[i]
    ax.scatter(X_house[:, i], y_house, s=12, alpha=0.35)
    corr = np.corrcoef(X_house[:, i], y_house)[0, 1]
    ax.set_xlabel(name)
    ax.set_ylabel("房价(万元)")
    ax.set_title(f"{name} vs 房价\nr = {corr:.3f}")
    ax.grid(alpha=0.3)
axes[-1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# 训练/测试划分 + 标准化 + 线性回归
X_train, X_test, y_train, y_test = train_test_split(X_house, y_house, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

multi_lr = LinearRegression().fit(X_train_scaled, y_train)
y_train_pred = multi_lr.predict(X_train_scaled)
y_test_pred = multi_lr.predict(X_test_scaled)

print(f"训练 R²: {r2_score(y_train, y_train_pred):.4f}")
print(f"测试 R²: {r2_score(y_test, y_test_pred):.4f}")
print(f"测试 RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.2f} 万元")

coef_table = sorted(zip(feature_names, multi_lr.coef_), key=lambda x: abs(x[1]), reverse=True)
print("\n标准化后的系数大小（可比较相对影响）：")
for name, coef in coef_table:
    print(f"{name:>10s}: {coef:+8.3f}")


### 8.1 为什么要特征缩放？

在多元线性回归中，不同特征的量纲可能差异很大：

- 面积：几十到几百
- 距地铁距离：0 到十几
- 绿化率：10 到 60

对梯度下降而言，特征尺度差异会让损失地形变成狭长的峡谷，导致训练路径来回震荡。

常见缩放：

- 标准化：$x'=(x-\mu)/\sigma$，均值 0、标准差 1
- Min-Max 归一化：$x'=(x-x_{min})/(x_{max}-x_{min})$，范围通常是 [0, 1]


In [ ]:
# 比较标准化和归一化的效果
std_scaler = StandardScaler()
mm_scaler = MinMaxScaler()
X_std = std_scaler.fit_transform(X_house)
X_mm = mm_scaler.fit_transform(X_house)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for i, name in enumerate(feature_names):
    axes[0].hist(X_house[:, i], bins=25, alpha=0.45, label=name)
    axes[1].hist(X_std[:, i], bins=25, alpha=0.45, label=name)
    axes[2].hist(X_mm[:, i], bins=25, alpha=0.45, label=name)

axes[0].set_title("原始尺度：不同特征不可直接比较")
axes[1].set_title("标准化：均值 0，标准差 1")
axes[2].set_title("归一化：压到 [0,1]")
for ax in axes:
    ax.grid(alpha=0.25)
axes[2].legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()

print("标准化后各列均值：", X_std.mean(axis=0).round(4))
print("标准化后各列标准差：", X_std.std(axis=0).round(4))


重要实践原则：

训练集和测试集不能分别 `fit` 缩放器。

正确做法：

1. 在训练集上 `fit`
2. 用同一个缩放器 `transform` 训练集和测试集

否则测试集的信息会泄露到预处理步骤，评估结果会过于乐观。


## 9. 批量、随机、小批量梯度下降

梯度下降可以根据每次使用多少样本分为三类：

| 方法 | 每次更新使用的数据 | 优点 | 缺点 |
|---|---|---|---|
| BGD 批量梯度下降 | 全部样本 | 稳定 | 大数据时慢 |
| SGD 随机梯度下降 | 1 个样本 | 单步快，可在线学习 | 震荡明显 |
| Mini-batch | 一小批样本 | 深度学习常用折中 | 需要选择 batch size |


In [ ]:
def train_gd_variant(x, y, method="batch", lr=1e-4, steps=800, batch_size=32):
    w, b = 0.0, 0.0
    losses = []
    n = len(x)
    local_rng = np.random.default_rng(123)
    for _ in range(steps):
        if method == "batch":
            idx = np.arange(n)
        elif method == "sgd":
            idx = local_rng.integers(0, n, size=1)
        elif method == "mini":
            idx = local_rng.choice(n, size=batch_size, replace=False)
        xb, yb = x[idx], y[idx]
        dw, db = gradients_for_line(xb, yb, w, b)
        w -= lr * dw
        b -= lr * db
        losses.append(mse_for_line(x, y, w, b))
    return w, b, np.array(losses)

results = {
    "BGD 全量": train_gd_variant(area, price, "batch", lr=1e-4, steps=1000),
    "SGD 单样本": train_gd_variant(area, price, "sgd", lr=1e-4, steps=1000),
    "Mini-batch": train_gd_variant(area, price, "mini", lr=1e-4, steps=1000, batch_size=4),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, (w, b, losses) in results.items():
    axes[0].plot(losses, label=name, alpha=0.85)
    axes[1].plot(x_line, w * x_line + b, label=f"{name}: w={w:.2f}")

axes[0].set_yscale("log")
axes[0].set_title("三种梯度下降的损失曲线")
axes[0].set_xlabel("更新次数")
axes[0].set_ylabel("MSE")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].scatter(area, price, s=45, alpha=0.6, color="gray", label="数据")
axes[1].set_title("训练后的拟合直线")
axes[1].set_xlabel("面积")
axes[1].set_ylabel("价格")
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.show()


## 10. 欠拟合、过拟合与多项式特征

线性回归不只能拟合直线，也可以配合多项式特征拟合曲线。

但模型复杂度过高时，会把噪声也记住，导致过拟合。

- 欠拟合：模型太简单，训练集也拟合不好
- 合适拟合：捕捉主要规律，但不过度追逐噪声
- 过拟合：训练集表现极好，测试集表现变差


In [ ]:
# 构造一个真实函数是 sin 的数据，比较不同多项式次数
rng_poly = np.random.default_rng(7)
x = np.sort(rng_poly.uniform(0, 1, 45))
y = np.sin(2 * np.pi * x) + rng_poly.normal(0, 0.18, size=x.size)
X_tr, X_te, y_tr, y_te = train_test_split(x.reshape(-1, 1), y, test_size=0.35, random_state=0)
x_dense = np.linspace(0, 1, 400).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, degree in zip(axes, [1, 5, 18]):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_tr, y_tr)
    train_mse = mean_squared_error(y_tr, model.predict(X_tr))
    test_mse = mean_squared_error(y_te, model.predict(X_te))
    ax.scatter(X_tr[:, 0], y_tr, s=35, label="训练", alpha=0.8)
    ax.scatter(X_te[:, 0], y_te, s=45, marker="x", label="测试", alpha=0.8)
    ax.plot(x_dense[:, 0], np.sin(2*np.pi*x_dense[:, 0]), "g--", lw=2, label="真实函数")
    ax.plot(x_dense[:, 0], model.predict(x_dense), "r", lw=2, label="模型")
    ax.set_title(f"degree={degree}\n训练MSE={train_mse:.3f}, 测试MSE={test_mse:.3f}")
    ax.set_ylim(-1.6, 1.6)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 10.1 正则化：限制模型不要太“任性”

正则化会在原来的 MSE 后面加惩罚项：

Ridge / L2：

$$Loss = MSE + \lambda \sum_j w_j^2$$

Lasso / L1：

$$Loss = MSE + \lambda \sum_j |w_j|$$

直觉：如果模型想用非常大的权重去追逐噪声，就要付出额外代价。


In [ ]:
# 在高次多项式上比较无正则、Ridge、Lasso
degree = 18
models = {
    "无正则化": make_pipeline(PolynomialFeatures(degree), LinearRegression()),
    "Ridge α=0.01": make_pipeline(PolynomialFeatures(degree), Ridge(alpha=0.01)),
    "Lasso α=0.001": make_pipeline(PolynomialFeatures(degree), Lasso(alpha=0.001, max_iter=20000)),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_tr, y_tr)
    train_mse = mean_squared_error(y_tr, model.predict(X_tr))
    test_mse = mean_squared_error(y_te, model.predict(X_te))
    ax.scatter(X_tr[:, 0], y_tr, s=35, alpha=0.75, label="训练")
    ax.scatter(X_te[:, 0], y_te, s=45, marker="x", alpha=0.75, label="测试")
    ax.plot(x_dense[:, 0], np.sin(2*np.pi*x_dense[:, 0]), "g--", lw=2, label="真实函数")
    ax.plot(x_dense[:, 0], model.predict(x_dense), "r", lw=2, label="模型")
    ax.set_title(f"{name}\n训练MSE={train_mse:.3f}, 测试MSE={test_mse:.3f}")
    ax.set_ylim(-1.6, 1.6)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 11. 用交叉验证选择模型复杂度

只划分一次训练集/测试集有随机性。交叉验证会把训练数据分成 K 份，轮流拿其中 1 份验证，其余 K-1 份训练。

这样能更稳定地估计不同模型复杂度的泛化表现。


In [ ]:
# 用交叉验证比较不同多项式次数
degrees = range(1, 16)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_means = []
cv_stds = []

for degree in degrees:
    model = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=0.001))
    # cross_val_score 越大越好；这里用 neg_mean_squared_error，所以取负号还原 MSE
    scores = -cross_val_score(model, x.reshape(-1, 1), y, cv=cv, scoring="neg_mean_squared_error")
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

best_degree = degrees[int(np.argmin(cv_means))]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(list(degrees), cv_means, yerr=cv_stds, marker="o", capsize=4)
ax.axvline(best_degree, color="r", ls="--", label=f"最佳 degree={best_degree}")
ax.set_xlabel("多项式次数")
ax.set_ylabel("5折交叉验证 MSE")
ax.set_title("用交叉验证选择复杂度")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

print("最佳多项式次数：", best_degree)


## 12. 完整项目：模拟房价预测流程

现在把前面的知识串起来，完成一个标准机器学习项目流程：

1. 准备数据
2. 探索性分析
3. 划分训练集/测试集
4. 特征缩放
5. 训练普通线性回归和 Ridge 回归
6. 评估模型
7. 残差诊断
8. 解释特征影响


In [ ]:
# 使用前面生成的多特征房价数据，封装成一个可复用的评估函数
def regression_report(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    return {
        "模型": name,
        "训练R2": r2_score(y_train, pred_train),
        "测试R2": r2_score(y_test, pred_test),
        "测试RMSE": np.sqrt(mean_squared_error(y_test, pred_test)),
        "测试MAE": mean_absolute_error(y_test, pred_test),
        "预测": pred_test,
        "模型对象": model,
    }

reports = []
reports.append(regression_report("LinearRegression", LinearRegression(), X_train_scaled, X_test_scaled, y_train, y_test))
for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
    reports.append(regression_report(f"Ridge α={alpha}", Ridge(alpha=alpha), X_train_scaled, X_test_scaled, y_train, y_test))

print(f"{'模型':<20} {'训练R2':>8} {'测试R2':>8} {'RMSE':>10} {'MAE':>10}")
print("-" * 62)
for r in reports:
    print(f"{r['模型']:<20} {r['训练R2']:>8.4f} {r['测试R2']:>8.4f} {r['测试RMSE']:>10.3f} {r['测试MAE']:>10.3f}")

best = max(reports, key=lambda r: r["测试R2"])
print("\n测试集 R² 最好的模型：", best["模型"])


In [ ]:
# 残差诊断和预测效果展示
best_pred = best["预测"]
best_resid = y_test - best_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(y_test, best_pred, s=35, alpha=0.65)
lo = min(y_test.min(), best_pred.min())
hi = max(y_test.max(), best_pred.max())
axes[0].plot([lo, hi], [lo, hi], "r--")
axes[0].set_xlabel("真实房价")
axes[0].set_ylabel("预测房价")
axes[0].set_title("真实值 vs 预测值")
axes[0].grid(alpha=0.3)

axes[1].scatter(best_pred, best_resid, s=35, alpha=0.65)
axes[1].axhline(0, color="r", ls="--")
axes[1].set_xlabel("预测房价")
axes[1].set_ylabel("残差")
axes[1].set_title("残差 vs 预测值")
axes[1].grid(alpha=0.3)

axes[2].hist(best_resid, bins=24, edgecolor="white", alpha=0.8)
axes[2].axvline(0, color="r", ls="--")
axes[2].set_xlabel("残差")
axes[2].set_ylabel("频数")
axes[2].set_title("残差分布")
axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 解释标准化后的特征系数
best_model = best["模型对象"]
coef = best_model.coef_
order = np.argsort(np.abs(coef))[::-1]

fig, ax = plt.subplots(figsize=(8, 4.8))
colors = ["tomato" if coef[i] < 0 else "steelblue" for i in order]
ax.barh(np.array(feature_names)[order], coef[order], color=colors, alpha=0.85)
ax.axvline(0, color="black", lw=0.8)
ax.invert_yaxis()
ax.set_xlabel("标准化特征系数")
ax.set_title("特征影响方向与强度（标准化后可比较）")
ax.grid(axis="x", alpha=0.3)
plt.show()

print("解释：正系数表示特征越大，预测房价越高；负系数表示特征越大，预测房价越低。")


## 13. 常见误区与检查清单

### 误区 1：训练集表现好就代表模型好

不一定。训练集好、测试集差，往往是过拟合。

### 误区 2：线性回归只能画直线

不完全对。线性回归对参数线性，但可以配合多项式、对数、交互项等特征变换，拟合非线性关系。

### 误区 3：系数绝对值越大，特征越重要

只有在特征尺度可比较时才成立。通常要先标准化，再比较系数。

### 误区 4：R² 永远在 0 到 1 之间

错误。R² 可以小于 0，表示模型比直接预测均值还差。

### 实战检查清单

1. 是否明确了目标变量和特征？
2. 是否划分了训练集和测试集？
3. 缩放器是否只在训练集上 fit？
4. 是否同时看了 RMSE/MAE/R²？
5. 是否画了残差图？
6. 如果模型表现差，是欠拟合、过拟合，还是特征不够？
7. 如果用了正则化，是否用验证集或交叉验证选择强度？


## 14. 本节总结与练习

线性回归的核心知识图谱：

```text
线性回归
├── 模型假设
│   ├── 一元：ŷ = wx + b
│   └── 多元：ŷ = Xw + b
├── 损失函数
│   ├── MSE：可导、凸、常用
│   ├── MAE：稳健但不可处处求导
│   └── RMSE/R²：更适合解释和评估
├── 参数求解
│   ├── 梯度下降：可扩展，需要学习率
│   └── 正规方程：小数据方便，数值问题要注意
├── 诊断
│   ├── 残差图
│   ├── 训练/测试误差
│   └── 欠拟合/过拟合
└── 改进
    ├── 特征工程
    ├── 特征缩放
    ├── 正则化 Ridge/Lasso
    └── 交叉验证
```

练习：

1. 把房价项目中的噪声标准差从 12 改为 30，观察 R² 和残差图如何变化。
2. 去掉“面积”特征重新训练，模型表现下降多少？为什么？
3. 给房价数据增加一个无关随机特征，普通线性回归和 Ridge 的系数有什么变化？
4. 在多项式例子中把 `Ridge(alpha=0.001)` 改成不同 alpha，观察最佳复杂度是否变化。
5. 自己实现多元线性回归的梯度下降版本，并与 `sklearn.LinearRegression` 对比。

下一节可以自然过渡到逻辑回归：当目标不再是连续数值，而是类别概率时，线性模型会怎样变化？
